<div align="center" style="font-size: 2.5em; font-weight: bold; margin-bottom: 10px;">ChatGTP</div>
<div align="center" style="font-size: 1.8em; font-weight: bold; margin-bottom: 15px;">Gdje to piše (DzeToPishe)</div>
<div align="center" style="font-size: 1.3em; margin-bottom: 25px;">Diplomski rad - Automatsko ocjenjivanje odgovora</div>
<div align="right" style="font-style: italic; color: #666;">by Zlatko Pračić</div>

## 1. Postavljanje okruženja

Ovaj dio notebooka instalira potrebne biblioteke za izračun metrika i uvozi sve module.

### Instalacija potrebnih biblioteka

Instaliramo biblioteke za automatsko ocjenjivanje:
- **rouge-score** - ROUGE-L metrika za najduži zajednički podniz
- **bert-score** - BERTScore za kontekstualnu semantičku sličnost
- **sentence-transformers** - embedding model za cosine similarity

In [ ]:
!pip install rouge-score bert-score
!pip install -U sentence-transformers

### Uvoz biblioteka

Uvozimo sve potrebne module na jednom mjestu za preglednost.

In [2]:
import pandas as pd
import numpy as np
import json
import logging

from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine

print('Sve biblioteke uspješno učitane.')

Sve biblioteke uspješno učitane.


## 2. Učitavanje podataka

Učitavamo rezultate generiranja iz CSV datoteke i referentne odgovore iz JSON datoteke. CSV sadrži odgovore sva tri pristupa: **Vanilla**, **FAISS RAG** i **GraphRAG**.

### Učitavanje CSV rezultata i JSON referenci

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
csv_path  = '/content/drive/My Drive/diplomskiRad/rezultati.csv'
json_path = '/content/drive/My Drive/diplomskiRad/pitanja_odgovori.json'

df = pd.read_csv(csv_path, encoding='utf-8-sig')

with open(json_path, 'r', encoding='utf-8') as f:
    qa_data = json.load(f)

# Obavezni stupci
potrebni_stupci = ['rbr', 'kategorija', 'pitanje', 'ocekivani_odgovor',
                   'vanilla_odgovor', 'rag_odgovor']
nedostaju = [s for s in potrebni_stupci if s not in df.columns]
if nedostaju:
    raise ValueError(f'Nedostaju stupci u CSV-u: {nedostaju}')

# GraphRAG stupac je opcionalan (može se dodati naknadno)
ima_graph_rag = 'graph_rag_odgovor' in df.columns
if ima_graph_rag:
    print('\u2713 GraphRAG stupac pronađen — evaluacija sva tri pristupa.')
else:
    print('\u26a0  GraphRAG stupac nije pronađen — evaluacija Vanilla i FAISS RAG.')

print(f'\nUčitano {len(df)} redaka iz CSV-a.')
print(f'Stupci: {list(df.columns)}')
print(f'\nKategorije:')
print(df['kategorija'].value_counts().to_string())
print(f'\nJSON metadata: {qa_data["metadata"]["ukupno_pitanja"]} pitanja')

✓ GraphRAG stupac pronađen — evaluacija sva tri pristupa.

Učitano 31 redaka iz CSV-a.
Stupci: ['rbr', 'kategorija', 'pitanje', 'ocekivani_odgovor', 'vanilla_odgovor', 'vanilla_vrijeme_sekunde', 'rag_odgovor', 'rag_vrijeme_sekunde', 'graph_rag_odgovor', 'graph_rag_vrijeme_sekunde']

Kategorije:
kategorija
jednostavna_pitanja                   23
kompleksna_pitanja_jedan_dokument      5
kompleksna_pitanja_vise_dokumenata     3

JSON metadata: 31 pitanja


## 3. Funkcije za izračun metrika

Definiramo 3 metrike za automatsko ocjenjivanje odgovora:
- **ROUGE-L** — leksička metrika, mjeri najduži zajednički podniz (F1)
- **BERTScore** — semantička metrika, kontekstualna sličnost tokena
- **Cosine Similarity** — semantička metrika, sličnost embedding vektora cijelih odgovora

In [5]:
def compute_rouge_l(reference, hypothesis):
    """Izračunava ROUGE-L F1 score između referentnog i generiranog odgovora.

    ROUGE-L mjeri najduži zajednički podniz (Longest Common Subsequence)
    i vraća F1 mjeru koja balansira preciznost i odziv.
    """
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
    scores = scorer.score(reference, hypothesis)
    return scores['rougeL'].fmeasure


def compute_bertscore(references, hypotheses):
    """Izračunava BERTScore za liste referenci i hipoteza (batch).

    Koristi bert-base-multilingual-cased koji podržava hrvatski jezik.
    Vraća F1 komponentu BERTScore-a.
    """
    P, R, F1 = bert_score_fn(
        hypotheses, references,
        model_type='bert-base-multilingual-cased',
        num_layers=9,
        verbose=True
    )
    return F1.tolist()


def compute_cosine_similarity(reference, hypothesis, model):
    """Izračunava cosine similarity između embeddinga referentnog i generiranog odgovora.

    Koristi multilingual-e5-large model za vektorizaciju cijelih odgovora.
    """
    emb_ref = model.encode([reference], convert_to_numpy=True)
    emb_hyp = model.encode([hypothesis], convert_to_numpy=True)
    return float(sklearn_cosine(emb_ref, emb_hyp)[0][0])


print('Funkcije za metrike definirane.')

Funkcije za metrike definirane.


## 4. Izračun metrika za sve odgovore

Za svaki od 31 redak računamo 3 metrike za sve prisutne pristupe (Vanilla, FAISS RAG, i GraphRAG ako je dostupan) u usporedbi s očekivanim (referentnim) odgovorom.

### Inicijalizacija embedding modela

In [ ]:
_transformers_logger = logging.getLogger('transformers.modeling_utils')
_prev_level = _transformers_logger.level
_transformers_logger.setLevel(logging.ERROR)

embed_model = SentenceTransformer('intfloat/multilingual-e5-large')

_transformers_logger.setLevel(_prev_level)
print('Embedding model učitan.')

### Izračun ROUGE-L i Cosine Similarity

In [7]:
vanilla_rouge_l = []
vanilla_cosine  = []
rag_rouge_l     = []
rag_cosine      = []
graph_rouge_l   = []
graph_cosine    = []

for i, row in df.iterrows():
    ref = str(row['ocekivani_odgovor'])
    van = str(row['vanilla_odgovor'])
    rag = str(row['rag_odgovor'])

    vanilla_rouge_l.append(compute_rouge_l(ref, van))
    vanilla_cosine.append(compute_cosine_similarity(ref, van, embed_model))
    rag_rouge_l.append(compute_rouge_l(ref, rag))
    rag_cosine.append(compute_cosine_similarity(ref, rag, embed_model))

    if ima_graph_rag:
        grph = str(row['graph_rag_odgovor'])
        graph_rouge_l.append(compute_rouge_l(ref, grph))
        graph_cosine.append(compute_cosine_similarity(ref, grph, embed_model))

    if (i + 1) % 10 == 0 or i == 0:
        print(f'Obrađeno {i + 1}/{len(df)} pitanja...')

df['vanilla_rouge_l'] = vanilla_rouge_l
df['vanilla_cosine']  = vanilla_cosine
df['rag_rouge_l']     = rag_rouge_l
df['rag_cosine']      = rag_cosine
if ima_graph_rag:
    df['graph_rouge_l'] = graph_rouge_l
    df['graph_cosine']  = graph_cosine

print(f'\nROUGE-L i Cosine Similarity izračunati za svih {len(df)} pitanja.')

Obrađeno 1/31 pitanja...
Obrađeno 10/31 pitanja...
Obrađeno 20/31 pitanja...
Obrađeno 30/31 pitanja...

ROUGE-L i Cosine Similarity izračunati za svih 31 pitanja.


### Izračun BERTScore (batch)

In [ ]:
references         = df['ocekivani_odgovor'].astype(str).tolist()
vanilla_hypotheses = df['vanilla_odgovor'].astype(str).tolist()
rag_hypotheses     = df['rag_odgovor'].astype(str).tolist()

print('Izračunavam BERTScore za vanilla odgovore...')
df['vanilla_bertscore'] = compute_bertscore(references, vanilla_hypotheses)

print('\nIzračunavam BERTScore za FAISS RAG odgovore...')
df['rag_bertscore'] = compute_bertscore(references, rag_hypotheses)

if ima_graph_rag:
    graph_hypotheses = df['graph_rag_odgovor'].astype(str).tolist()
    print('\nIzračunavam BERTScore za GraphRAG odgovore...')
    df['graph_bertscore'] = compute_bertscore(references, graph_hypotheses)

print(f'\nBERTScore izračunat za svih {len(df)} pitanja.')

### Spremanje proširenog CSV-a

In [9]:
output_csv = '/content/drive/My Drive/diplomskiRad/rezultati_evaluacija.csv'
df.to_csv(output_csv, index=False, encoding='utf-8-sig')

print(f'Rezultati spremljeni u: {output_csv}')
print(f'Oblik: {df.shape[0]} redaka × {df.shape[1]} stupaca')
print(f'\nStupci: {list(df.columns)}')

# Pregled prvih redaka s metrikama
pregled_stupci = ['rbr', 'kategorija',
                  'vanilla_rouge_l', 'vanilla_bertscore', 'vanilla_cosine',
                  'rag_rouge_l',     'rag_bertscore',     'rag_cosine']
if ima_graph_rag:
    pregled_stupci += ['graph_rouge_l', 'graph_bertscore', 'graph_cosine']
df[pregled_stupci].head()

Rezultati spremljeni u: /content/drive/My Drive/diplomskiRad/rezultati_evaluacija.csv
Oblik: 31 redaka × 19 stupaca

Stupci: ['rbr', 'kategorija', 'pitanje', 'ocekivani_odgovor', 'vanilla_odgovor', 'vanilla_vrijeme_sekunde', 'rag_odgovor', 'rag_vrijeme_sekunde', 'graph_rag_odgovor', 'graph_rag_vrijeme_sekunde', 'vanilla_rouge_l', 'vanilla_cosine', 'rag_rouge_l', 'rag_cosine', 'graph_rouge_l', 'graph_cosine', 'vanilla_bertscore', 'rag_bertscore', 'graph_bertscore']


,rbr,kategorija,vanilla_rouge_l,vanilla_bertscore,vanilla_cosine,rag_rouge_l,rag_bertscore,rag_cosine,graph_rouge_l,graph_bertscore,graph_cosine
0,1,kompleksna_pitanja_vise_dokumenata,0.108787,0.690926,0.916388,0.141379,0.681684,0.909166,0.168103,0.714348,0.919309
1,2,kompleksna_pitanja_vise_dokumenata,0.113333,0.661969,0.916355,0.244514,0.680676,0.925577,0.138138,0.634464,0.920748
2,3,kompleksna_pitanja_vise_dokumenata,0.116022,0.668536,0.893689,0.224390,0.678575,0.931883,0.181818,0.653797,0.924337
3,4,kompleksna_pitanja_jedan_dokument,0.194175,0.681242,0.904260,0.203474,0.670209,0.889591,0.275281,0.688700,0.889881
4,5,kompleksna_pitanja_jedan_dokument,0.169279,0.649679,0.920220,0.151042,0.631718,0.919474,0.143322,0.616432,0.915345


## 5. Nastavak analize u R-u

Agregacija rezultata po kategorijama pitanja te sve tri vizualizacije (usporedba pristupa po metrikama, metrike po kategorijama, toplinska karta po pitanjima) više se ne rade u ovoj bilježnici, već u `statistika/skripta_2_metrike_llm.R` (odjeljak 9), koji učitava upravo `rezultati_evaluacija.csv` spremljen u prethodnom koraku. Time se izbjegava dupliciranje iste logike agregacije/vizualizacije na dva mjesta (Python i R) — ova bilježnica sada radi isključivo ono što se ne može odraditi u R-u bez oslanjanja na Python (izračun ROUGE-L, BERTScore i cosine sličnosti iz sirovog teksta pomoću transformer modela).